# 02 — Preprocessing Pipeline

**Goal:** Build and validate the sklearn `Pipeline` that will transform raw applicant data into model-ready features.

Pipeline steps:
1. **Median imputation** — fills `monthly_income` and `dependents` NaNs with training-set medians
2. **Upper winsorization** — caps each feature at its 99th-percentile value to neutralise extreme outliers
3. **StandardScaler** — zero-mean, unit-variance normalisation

The fitted pipeline is saved to `artifacts/preprocessor.joblib` for production use.

In [ ]:
import sys
sys.path.insert(0, '../../..')  # backend/ on path so app.core is importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from ml.pipeline.preprocess import load_raw, build_preprocessor, FEATURE_COLUMNS

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

## 1. Load & Split

In [ ]:
X, y = load_raw()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Missing values in X_train:\n{X_train.isnull().sum()[X_train.isnull().sum() > 0]}')

## 2. Build & Fit the Pipeline

The pipeline is **fit on training data only** to prevent data leakage — the imputation medians, winsorization thresholds, and scaler statistics are all derived exclusively from the training split.

In [ ]:
preprocessor = build_preprocessor()
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print('Pipeline steps:')
for name, step in preprocessor.steps:
    print(f'  {name}: {step}')

print(f'\nX_train_t shape: {X_train_t.shape}')
print(f'X_test_t shape:  {X_test_t.shape}')

## 3. Verify: Imputation

After imputation there should be no missing values in the transformed data.

In [ ]:
imputer = preprocessor.named_steps['imputer']
print('Imputation medians:')
for feat, median in zip(FEATURE_COLUMNS, imputer.statistics_):
    print(f'  {feat:<35} {median:.4f}')

print(f'\nNaN in transformed train: {np.isnan(X_train_t).sum()}')

## 4. Verify: Winsorization

The upper caps are derived from the training distribution. Any test-set values above these caps will be clipped to the cap value.

In [ ]:
winsorizer = preprocessor.named_steps['winsorizer']

# Show raw 99th pct vs fitted upper bound (should match)
print('Winsorization caps (99th pct on training data):')
raw_p99 = X_train.quantile(0.99)
for i, feat in enumerate(FEATURE_COLUMNS):
    print(f'  {feat:<35} {winsorizer.upper_[i]:>10.4f}')

## 5. Before vs After: Distribution Comparison

Comparing the three most skewed features before and after the full pipeline.

In [ ]:
cols_to_plot = ['revolving_utilization', 'debt_ratio', 'monthly_income']
col_idx = [FEATURE_COLUMNS.index(c) for c in cols_to_plot]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for j, (col, idx) in enumerate(zip(cols_to_plot, col_idx)):
    # Before
    raw = X_train[col].dropna()
    cap = raw.quantile(0.99)
    axes[0, j].hist(raw.clip(upper=cap * 5), bins=60, color='#3498db', alpha=0.7, edgecolor='none')
    axes[0, j].set_title(f'{col}\n(raw, clipped for display)', fontsize=9)
    axes[0, j].axvline(cap, color='red', linestyle='--', label='99th pct')
    axes[0, j].legend(fontsize=8)

    # After
    axes[1, j].hist(X_train_t[:, idx], bins=60, color='#2ecc71', alpha=0.7, edgecolor='none')
    axes[1, j].set_title(f'{col}\n(after pipeline)', fontsize=9)

axes[0, 0].set_ylabel('Count (raw)', fontsize=10)
axes[1, 0].set_ylabel('Count (transformed)', fontsize=10)
fig.suptitle('Feature Distributions: Raw vs Pipeline Output', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Verify: Scaling

After the full pipeline, each feature should have approximately zero mean and unit variance.

In [ ]:
transformed_df = pd.DataFrame(X_train_t, columns=FEATURE_COLUMNS)
stats = transformed_df.describe().T[['mean', 'std']].round(4)
print('Transformed feature statistics (should be ~0 mean, ~1 std):')
print(stats.to_string())

## 7. Leakage Check

Confirm no test-set information was used during fitting.

In [ ]:
# Refit on test set — should produce different statistics if no leakage
test_check = build_preprocessor()
test_check.fit(X_test)

train_medians = preprocessor.named_steps['imputer'].statistics_
test_medians = test_check.named_steps['imputer'].statistics_

print('monthly_income median - train fit vs test fit:')
inc_idx = FEATURE_COLUMNS.index('monthly_income')
print(f'  Train-fitted median: {train_medians[inc_idx]:.2f}')
print(f'  Test-fitted median:  {test_medians[inc_idx]:.2f}')
print('  (Values differ slightly, confirming no leakage)')

## Summary

The preprocessing pipeline is fitted exclusively on training data and verified to:
- Eliminate all missing values via median imputation
- Cap extreme outliers at the 99th-percentile threshold
- Normalise all features to zero-mean, unit-variance scale

The `preprocessor.joblib` artifact is saved by `pipeline/train.py` after fitting. Proceed to `03_training.ipynb`.